In [ ]:
!pip install --upgrade openai


In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Evaluate two baselines on your conversation-style test set:
  1) Qwen2.5-VL (HF transformers)
  2) GPT-5 (OpenAI API; image via base64 data URIs)

Input format: one JSON object per line with:
{
  "image": "path" OR ["path1","path2",...],
  "conversations": [
    {"from":"human","value":"... <image> ..."},
    {"from":"gpt","value":"...","actions":[...], ...},
    ...
  ]
}

We DO NOT execute tools. We simply feed the conversation (including
the *human-injected tool outputs*) and ask each model to answer the
earliest explicit question concisely.

Outputs:
  - outputs/{model_name}.jsonl   # per-sample predictions + light eval
  - outputs/summary.json         # macro averages of light eval metrics

Run:
  OPENAI_API_KEY=... python eval_qwen_and_gpt5.py \
      --input test.jsonl \
      --qwen_id Qwen/Qwen2.5-VL-7B-Instruct \
      --out_dir outputs
"""

import os, io, re, json, base64, argparse, math
from typing import List, Dict, Any, Optional, Tuple
from collections import defaultdict
from PIL import Image

# ---------- Small helpers ----------
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip().lower()

def strip_punct(s: str) -> str:
    return re.sub(r"[^\w\s]", " ", s or "")

def tokenize(s: str) -> List[str]:
    return [t for t in strip_punct(norm(s)).split() if t]

def jaccard_tokens(a: str, b: str) -> float:
    A, B = set(tokenize(a)), set(tokenize(b))
    if not A and not B: return 1.0
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)

def detect_irrelevant_blob(answer: str) -> int:
    a = norm(answer)
    if len(a) > 1200 and ("genome synteny" in a or "web-based" in a):
        return 1
    return 0

def find_first_question(convos: List[Dict[str, Any]]) -> Optional[str]:
    for t in convos:
        if t.get("from") == "human":
            val = t.get("value", "")
            vnorm = norm(val)
            if ("?" in val) or any(k in vnorm for k in ["what", "can you", "please", "examine", "identify"]):
                return val
    return None


In [3]:
# ---------- Conversation → model-specific message builders ----------

def _collect_images(sample_image_field, conversations) -> List[str]:
    if isinstance(sample_image_field, list):
        return [os.path.join(BASE_DIR, p) if not os.path.isabs(p) else p
                for p in sample_image_field]
    elif isinstance(sample_image_field, str):
        return [os.path.join(BASE_DIR, sample_image_field) if not os.path.isabs(sample_image_field) else sample_image_field]
    else:
        return []

# def _collect_images(sample_image_field, conversations) -> List[str]:
#     """
#     Return a list of image paths. If sample["image"] is a list, return that.
#     If it's a single path, wrap into a list. If messages contain <image>, we
#     just map occurrences to these paths in order; if fewer placeholders than
#     images, we attach extras to the last user turn.
#     """
#     if isinstance(sample_image_field, list):
#         return sample_image_field
#     elif isinstance(sample_image_field, str):
#         return [sample_image_field]
#     else:
#         # try to discover any file-looking tokens in human text (rare)
#         return []

def _split_on_image_tokens(text: str) -> List[str]:
    # Split text where '<image>' placeholders appear, keeping surrounding text.
    parts = [p for p in re.split(r"\s*<image>\s*", text)]
    return parts

def build_qwen_messages(sample: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Build Qwen2.5-VL 'messages' list with image items:
    [
      {"role":"user","content":[{"type":"image","image":"file:///path"}, {"type":"text","text":"..."}]},
      ...
    ]

    We include only HUMAN messages (and their tool outputs), ignoring GPT turns.
    """
    images = _collect_images(sample.get("image"), sample.get("conversations", []))
    img_idx = 0
    messages = []

    # System instruction to ensure concise, clinically oriented final answer
    sys_text = ("You are a medical AI assistant. "
                "Use any tool outputs provided *in the conversation text* as evidence. "
                "Answer the earliest explicit question concisely and clinically. "
                "Do not fabricate tools or calls.")
    messages.append({"role": "system", "content": sys_text})

    for turn in sample.get("conversations", []):
        if turn.get("from") != "human":
            continue
        val = turn.get("value", "")
        # If the human inserted <image> placeholders, attach images accordingly.
        parts = _split_on_image_tokens(val)
        content = []
        for i, part in enumerate(parts):
            # attach text chunk
            if part.strip():
                content.append({"type": "text", "text": part.strip()})
            # after each part except the last, attach an image (if available)
            if i < len(parts) - 1 and img_idx < len(images):
                path = images[img_idx]
                # Qwen quickstart accepts file:// URLs or http(s) URLs for images
                # (per model card + utils)  ➜ we use file:// scheme.
                content.append({"type": "image", "image": f"file://{os.path.abspath(path)}"})
                img_idx += 1

        # If the human message had no <image> but we still have unused images and
        # the message *looks* like it's referencing an image, attach one.
        if "<image>" not in val and img_idx < len(images) and ("image" in norm(val) or "x-ray" in norm(val) or "ct" in norm(val)):
            content = [{"type": "image", "image": f"file://{os.path.abspath(images[img_idx])}"}] + content
            img_idx += 1

        # Ensure there's at least one content piece
        if not content:
            content = [{"type": "text", "text": val.strip()}]

        messages.append({"role": "user", "content": content})

    return messages

def _image_to_data_uri(path: str) -> str:
    with open(path, "rb") as f:
        b = f.read()
    # Guess MIME from extension (fallback to png)
    ext = os.path.splitext(path)[1].lower()
    mime = "image/png"
    if ext in [".jpg", ".jpeg"]:
        mime = "image/jpeg"
    elif ext in [".webp"]:
        mime = "image/webp"
    return f"data:{mime};base64," + base64.b64encode(b).decode("utf-8")

def build_openai_messages(sample: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Build OpenAI Chat Completions messages with vision content:
    messages = [
      {"role":"system","content":"..."},
      {"role":"user","content":[{"type":"text","text":"..."},{"type":"image_url","image_url":{"url": data_uri}}]}
    ]
    """
    images = _collect_images(sample.get("image"), sample.get("conversations", []))
    img_idx = 0
    messages = []

    sys_text = ("You are a medical AI assistant. "
                "Use any tool outputs provided *in the conversation text* as evidence. "
                "Answer the earliest explicit question concisely and clinically. "
                "Do not fabricate tools or calls.")
    messages.append({"role": "system", "content": sys_text})

    for turn in sample.get("conversations", []):
        if turn.get("from") != "human":
            continue
        val = turn.get("value", "")
        parts = _split_on_image_tokens(val)
        content = []
        for i, part in enumerate(parts):
            if part.strip():
                content.append({"type": "text", "text": part.strip()})
            if i < len(parts) - 1 and img_idx < len(images):
                data_uri = _image_to_data_uri(images[img_idx])
                content.append({"type": "image_url", "image_url": {"url": data_uri}})
                img_idx += 1

        if "<image>" not in val and img_idx < len(images) and ("image" in norm(val) or "x-ray" in norm(val) or "ct" in norm(val)):
            data_uri = _image_to_data_uri(images[img_idx])
            content = [{"type": "image_url", "image_url": {"url": data_uri}}] + content
            img_idx += 1

        if not content:
            content = [{"type": "text", "text": val.strip()}]

        messages.append({"role": "user", "content": content})

    return messages



In [4]:
# ---------- Model runners ----------

class QwenRunner:
    """
    Qwen2.5-VL via HF transformers + qwen-vl-utils (per official quickstart).
    Ref: model card quickstart shows apply_chat_template + process_vision_info. 
    """
    def __init__(self, model_id: str, device_map: str = "auto", torch_dtype: str = "auto",
                 attn_impl: Optional[str] = None, max_new_tokens: int = 256):
        import torch
        from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
        from qwen_vl_utils import process_vision_info  # noqa

        self.torch = torch
        self.model_id = model_id
        self.max_new_tokens = max_new_tokens
        kwargs = {"torch_dtype": torch_dtype, "device_map": device_map}
        if attn_impl:
            kwargs["attn_implementation"] = attn_impl
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, **kwargs)
        self.processor = AutoProcessor.from_pretrained(model_id)

        # stash symbol to avoid reimport
        self._process_vision_info = __import__("qwen_vl_utils").process_vision_info

    def generate(self, messages: List[Dict[str, Any]]) -> str:
        # build text prompt and vision inputs per official quickstart
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = self._process_vision_info(messages)  # hf model card shows this path
        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to(self.model.device)
        with self.torch.inference_mode():
            out_ids = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens)
        # trim the prompt tokens
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
        out_text = self.processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        return out_text.strip()



In [5]:
class GPT5Runner:
    """
    GPT-5 via OpenAI Chat Completions API (vision messages with base64 data URIs).
    Assumes OPENAI_API_KEY is set. You can customize 'model' if needed.
    """
    def __init__(self, model: str = "gpt-5", temperature: float = 0.0, max_tokens: int = 512):
        from openai import OpenAI
        self.client = OpenAI()
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens

    def generate(self, messages: List[Dict[str, Any]]) -> str:
        # Convert our "content":[...] dicts into the expected schema (already aligned)
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=self.temperature,
            max_tokens=self.max_tokens,
        )
        return (resp.choices[0].message.content or "").strip()





In [6]:
# ---------- Light evaluation on outputs ----------

def light_eval(sample: Dict[str, Any], answer: str) -> Dict[str, float]:
    first_q = find_first_question(sample.get("conversations", [])) or ""
    return {
        "answers_first_question_jaccard": round(jaccard_tokens(first_q, answer), 4),
        "irrelevant_content_flag": int(detect_irrelevant_blob(answer)),
        "answer_len": len(answer),
    }


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = ""

TEST_JSONL = "multi_round/MMedAgent-V2/test/test.jsonl"
BASE_DIR = "multi_round"

OUT_DIR = "outputs_baseline"

QWEN_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

GPT5_MODEL = "gpt-5"
MAX_NEW_TOKENS = 512




In [8]:
import json, os
from collections import defaultdict

os.makedirs(OUT_DIR, exist_ok=True)
out_qwen = os.path.join(OUT_DIR, "qwen2_5_vl.jsonl")
out_gpt5 = os.path.join(OUT_DIR, "gpt5.jsonl")

qwen = QwenRunner(QWEN_ID, device_map="auto", torch_dtype="auto", attn_impl=None, max_new_tokens=MAX_NEW_TOKENS)
gpt5 = GPT5Runner(model=GPT5_MODEL, temperature=0.0, max_tokens=MAX_NEW_TOKENS)

summaries = defaultdict(list)

def run_model(runner, builder, out_path, tag):
    with open(TEST_JSONL, "r", encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for i, line in enumerate(fin, 1):
            line = line.strip()
            if not line:
                continue
            sample = json.loads(line)
            if "id" not in sample:
                sid = sample.get("image")
                if isinstance(sid, list):
                    sid = ",".join(sid)
                sample["id"] = sid or f"sample_{i}"

            messages = builder(sample)
            try:
                ans = runner.generate(messages)
            except Exception as e:
                ans = f"[ERROR during generation: {e}]"

            ev = light_eval(sample, ans)
            rec = {"id": sample["id"], "model": tag, "answer": ans, **ev}
            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
            for k, v in ev.items():
                if isinstance(v, (int, float)):
                    summaries[f"{tag}:{k}"].append(float(v))

run_model(qwen, build_qwen_messages, out_qwen, "qwen2_5_vl")
run_model(gpt5, build_openai_messages, out_gpt5, "gpt5")

print("Done. Files written to:", out_qwen, "and", out_gpt5)


/home/jack/anaconda3/envs/qwenvl-new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.40it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Done. Files written to: outputs_baseline/qwen2_5_vl.jsonl and outputs_baseline/gpt5.jsonl


In [10]:
import json
from statistics import mean
from collections import defaultdict

files = [
    "outputs_baseline/qwen2_5_vl.jsonl",
    "outputs_baseline/gpt5.jsonl"
]

summaries = defaultdict(list)

for path in files:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            for k, v in rec.items():
                if isinstance(v, (int, float)):
                    summaries[f"{rec['model']}:{k}"].append(v)

summary = {}
for k, vals in summaries.items():
    if vals:
        summary[k] = round(mean(vals), 4)

print("=== SUMMARY (macro-average) ===")
for k, v in summary.items():
    print(f"{k}: {v}")


=== SUMMARY (macro-average) ===
qwen2_5_vl:answers_first_question_jaccard: 0.1192
qwen2_5_vl:irrelevant_content_flag: 0
qwen2_5_vl:answer_len: 474.6867
gpt5:answers_first_question_jaccard: 0.0113
gpt5:irrelevant_content_flag: 0
gpt5:answer_len: 206


In [11]:
import pandas as pd, json

dfs = []
for path in [out_qwen, out_gpt5]:
    rows = [json.loads(x) for x in open(path, "r", encoding="utf-8")]
    dfs.append(pd.DataFrame(rows))
pd.concat(dfs, ignore_index=True)


,id,model,answer,answers_first_question_jaccard,irrelevant_content_flag,answer_len
0,"tool_image/test/plaquenil-toxicity-46.jpg,tool...",qwen2_5_vl,Based on the information provided and authorit...,0.0312,0,1213
1,tool_image/test/234da21f-200d4882-21527f23-609...,qwen2_5_vl,Coliform bacteria isolated from recreational l...,0.0189,0,284
2,tool_image/test/c5c6d1a7-12ad7b72-98a129a3-66f...,qwen2_5_vl,"The medical report indicates that the lungs, h...",0.0150,0,1418
3,"tool_image/test/dummy_img.png,tool_image/test/...",qwen2_5_vl,Glucagonoma is a rare neuroendocrine tumor tha...,0.0490,0,547
4,tool_image/test/group69-6.jpg,qwen2_5_vl,The patient has undergone a minimally invasive...,0.0183,0,1035
...,...,...,...,...,...,...
295,tool_image/test/BrEaST-Lesions_USG-images_and_...,gpt5,[ERROR during generation: Error code: 403 - {'...,0.0000,0,206
296,tool_image/test/WORD_0061_0098.jpg,gpt5,[ERROR during generation: Error code: 403 - {'...,0.0000,0,206
297,tool_image/test/WORD_0020_0070.jpg,gpt5,[ERROR during generation: Error code: 403 - {'...,0.0000,0,206
298,tool_image/test/juvenile-retinoschisis-28.jpg,gpt5,[ERROR during generation: Error code: 403 - {'...,0.0000,0,206


In [ ]:
# ---------- Main ----------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, help="Path to test.jsonl")
    ap.add_argument("--out_dir", default="outputs")
    ap.add_argument("--qwen_id", default="Qwen/Qwen2.5-VL-7B-Instruct")
    ap.add_argument("--gpt5_model", default="gpt-5")
    ap.add_argument("--max_new_tokens", type=int, default=256)
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    out_qwen = os.path.join(args.out_dir, "qwen2_5_vl.jsonl")
    out_gpt5 = os.path.join(args.out_dir, "gpt5.jsonl")

    # Init runners
    qwen = QwenRunner(args.qwen_id, device_map="auto", torch_dtype="auto",
                      attn_impl=None, max_new_tokens=args.max_new_tokens)
    gpt5 = GPT5Runner(model=args.gpt5_model, temperature=0.0, max_tokens=args.max_new_tokens)

    summaries = defaultdict(list)

    def run_model(runner, builder, out_path, tag):
        with open(args.input, "r", encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
            for line in fin:
                line = line.strip()
                if not line:
                    continue
                sample = json.loads(line)
                if "id" not in sample:
                    # derive an id for convenience
                    sid = sample.get("image")
                    if isinstance(sid, list):
                        sid = ",".join(sid)
                    sample["id"] = sid or f"sample_{len(summaries['dummy'])+1}"

                messages = builder(sample)
                try:
                    ans = runner.generate(messages)
                except Exception as e:
                    ans = f"[ERROR during generation: {e}]"

                ev = light_eval(sample, ans)
                rec = {
                    "id": sample["id"],
                    "model": tag,
                    "answer": ans,
                    **ev
                }
                fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
                # accumulate for macro avg
                for k, v in ev.items():
                    if isinstance(v, (int, float)):
                        summaries[f"{tag}:{k}"].append(float(v))

    # Qwen2.5-VL
    run_model(qwen, build_qwen_messages, out_qwen, "qwen2_5_vl")

    # GPT-5
    run_model(gpt5, build_openai_messages, out_gpt5, "gpt5")

    # Macro averages → summary.json
    summary = {}
    for k, vals in summaries.items():
        if vals:
            summary[k] = round(sum(vals)/len(vals), 4)

    with open(os.path.join(args.out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print("=== Done ===")
    print(f"Per-sample (Qwen): {out_qwen}")
    print(f"Per-sample (GPT-5): {out_gpt5}")
    print(f"Summary: {os.path.join(args.out_dir, 'summary.json')}")

if __name__ == "__main__":
    main()
